In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import ast
from sklearn.decomposition import PCA

# =========================================================
# Config
# =========================================================
INPUT_CSV = "movies_enriched.csv"
OUTPUT_CSV = "movies_overview_embeddings.csv"

ID_COL = "movieId"
OVERVIEW_COL = "overview"
ACTORS_COL = "actors"

MAX_ACTORS = 5
FALLBACK_TEXT = "No description available."

MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
BATCH_SIZE = 64

EMB_DIM = 768
PCA_DIM = 64
PCA_WHITEN = False

# =========================================================
# Load data
# =========================================================
df = pd.read_csv(INPUT_CSV)

for col in [ID_COL, OVERVIEW_COL]:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found in {INPUT_CSV}")

df[OVERVIEW_COL] = df[OVERVIEW_COL].fillna("")
df[ACTORS_COL] = df.get(ACTORS_COL, "").fillna("")

# =========================================================
# Helpers
# =========================================================
def parse_actors(val):
    if not val:
        return []

    if isinstance(val, list):
        return val[:MAX_ACTORS]

    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return parsed[:MAX_ACTORS]
        except Exception:
            pass

        return [a.strip() for a in val.split(",") if a.strip()][:MAX_ACTORS]

    return []

def build_text(row):
    text = row[OVERVIEW_COL].strip()

    actors = parse_actors(row[ACTORS_COL])
    if actors:
        text += "\n\nStarring: " + ", ".join(actors)

    if not text:
        text = FALLBACK_TEXT

    return text

# =========================================================
# Build texts
# =========================================================
texts = df.apply(build_text, axis=1).tolist()
assert len(texts) == len(df)

# =========================================================
# Load model
# =========================================================
model = SentenceTransformer(MODEL_NAME)

# =========================================================
# Encode (normalized)
# =========================================================
embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Encoding"):
    batch = texts[i:i + BATCH_SIZE]
    emb = model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    embeddings.append(emb)

embeddings = np.vstack(embeddings).astype(np.float32)

# =========================================================
# Safety checks
# =========================================================
assert embeddings.shape == (len(df), EMB_DIM)
assert np.isfinite(embeddings).all()

# =========================================================
# PCA
# =========================================================
pca = PCA(
    n_components=PCA_DIM,
    whiten=PCA_WHITEN,
    random_state=42
)

embeddings_pca = pca.fit_transform(embeddings).astype(np.float32)

# Renormalize after PCA (important for LSH)
norms = np.linalg.norm(embeddings_pca, axis=1, keepdims=True)
embeddings_pca = embeddings_pca / np.clip(norms, 1e-12, None)

# PCA checks
assert embeddings_pca.shape == (len(df), PCA_DIM)
assert np.isfinite(embeddings_pca).all()

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

# =========================================================
# Save (Spark-friendly wide CSV)
# =========================================================
emb_df = pd.DataFrame(
    embeddings_pca,
    columns=[f"emb_{i}" for i in range(PCA_DIM)]
)

out_df = pd.concat(
    [df[[ID_COL]].reset_index(drop=True), emb_df],
    axis=1
)

out_df.to_csv(OUTPUT_CSV, index=False)

# =========================================================
# Final confirmation
# =========================================================
print(f"Saved embeddings to: {OUTPUT_CSV}")
print(f"Rows: {out_df.shape[0]}")
print(f"Embedding dim: {PCA_DIM}")
print("✓ No nulls")
print("✓ No NaNs")
print("✓ One embedding per movie")

Encoding: 100%|██████████| 60/60 [00:34<00:00,  1.75it/s]


PCA explained variance: 0.639
Saved embeddings to: movies_overview_embeddings.csv
Rows: 3823
Embedding dim: 64
✓ No nulls
✓ No NaNs
✓ One embedding per movie
